# Решения при изменении параметров уравнения

Как накачка $F$ и отстройка $\Delta$ меняют распределение однородных решений, одиночных солитонов и прочих состояний при $T=100$?
Сравниваем $F=1.8,2,2.2$ при $\Delta=3$ и $\Delta=2.5,3,3.5$ при $F=2$.
В каждом случае восемь амплитуд $A\in[0.4,3.2]$ расположены равномерно,
восемь ширин $\sigma\in[0.05,0.7]$ — логарифмически.
Начальное поле — вещественная гауссиана без фона;
$\beta_2=-0.1$, $L=2\pi$, $N=512$, $\Delta t=0.0005$.

Сравнение проводится при $T=100$, включая запуски с более длительной эволюцией.
Цвет относится к отмеченному узлу; границы цветных ячеек не являются
уточнёнными границами бассейнов притяжения.

In [ ]:
import csv
from collections import Counter
from enum import Enum, auto
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch
from lle_runs.project import get_experiment_dir
from lle_runs.runs import get_run
from lle_solver.simulations.gaussian import GaussianConfig
from lle_solver.simulations.storage import load_evolution
from lle_analysis.fields import time_mask, relative_field_l2_error, spatial_contrast
from lle_analysis.peaks import peak_indices
from lle_analysis.criteria import (
    DEFAULT_STATIONARY_TOLERANCE, DEFAULT_HOMOGENEOUS_CONTRAST, is_stationary, is_homogeneous,
)
from lle_viz.grids import rectangular_grid
from lle_viz.plots import cell_edges, plot_outcome_map, plot_stacked_counts
from lle_viz.style import configure_plots, save_figure, format_log_x

In [ ]:
EXPERIMENT_DIR = get_experiment_dir(3)
RESULTS_DIR = EXPERIMENT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
configure_plots()

In [ ]:
with (EXPERIMENT_DIR / "baseline_runs.csv").open(encoding="utf-8") as source:
    baseline_selected = list(csv.DictReader(source))

In [ ]:
with (EXPERIMENT_DIR / "map_runs.csv").open(encoding="utf-8") as source:
    map_selected = list(csv.DictReader(source))

### Три категории на интервале $[80,100]$

- **Однородное решение:** комплексное поле стационарно, интенсивность однородна на каждом снимке.
- **Одиночный солитон:** комплексное поле стационарно, интенсивность неоднородна и имеет ровно один выраженный пик на каждом снимке.
- **Прочее:** остальные состояния; категория не определяет характер динамики.


In [ ]:
class SolutionKind(Enum):
    HOMOGENEOUS = auto()
    SINGLE_SOLITON = auto()
    OTHER = auto()

solution_labels = {
    SolutionKind.HOMOGENEOUS: "Однородное решение",
    SolutionKind.SINGLE_SOLITON: "Одиночный солитон",
    SolutionKind.OTHER: "Прочее",
}
solutions = list(solution_labels)
colors = ["#293142", "#4fc48b", "#b580df"]

In [ ]:
map_results = []
profiles_by_case = {}
observation_time = 100.
observation_duration = 20.
profile_times = (0., 20., 50., 100.)
for row in baseline_selected + map_selected:
    run = get_run(row["run_id"])
    config = run.load_config(GaussianConfig)
    evolution = load_evolution(run.path)
    tail = evolution[time_mask(evolution.times, observation_time-observation_duration, observation_time)]
    if (tail.times.size < 2
            or not np.isclose(tail.times[0], 80., rtol=0, atol=1e-10)
            or not np.isclose(tail.times[-1], 100., rtol=0, atol=1e-10)):
        raise ValueError(f"Запуск {run.id} не покрывает окно [80,100]")
    intensity = tail.intensity
    stationary = is_stationary(tail)
    solution = SolutionKind.OTHER
    if stationary:
        if all(is_homogeneous(profile) for profile in intensity):
            solution = SolutionKind.HOMOGENEOUS
        elif all(not is_homogeneous(profile) and len(peak_indices(profile)) == 1
                 for profile in intensity):
            solution = SolutionKind.SINGLE_SOLITON
    map_results.append(dict(case=row["case"], run_id=run.id, A=round(config.initial.amplitude, 12),
                            sigma=round(config.initial.width, 12), F=config.equation.pump_amplitude,
                            delta=config.equation.detuning, solution=solution,
                            relative_field_l2_error=float(relative_field_l2_error(tail, tail[-1]).max()),
                            contrast=float(spatial_contrast(intensity).max())))
    profiles_by_case.setdefault(row["case"], {})[round(config.initial.amplitude, 12), round(config.initial.width, 12)] = [
        evolution.at_time(time).intensity for time in profile_times
    ]


In [ ]:
case_order = ["baseline", "F_lower", "F_higher", "delta_lower", "delta_higher"]

## F=2, Δ=3

Порядок рисунков: изменение поля и контраст; три категории решений; начальная энергия; профили при $t=0,20,50,100$. Цвета, маркеры и шкалы совпадают с экспериментом 1. Энергия вычисляется аналитически: $E_0=\sqrt{\pi}A^2\sigma$.


In [ ]:
case = 'baseline'
case_title = 'F=2, Δ=3'
coarse = [row for row in map_results if row["case"] == case]
measurements = coarse

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True, layout="constrained")
metrics = (
    ("relative_field_l2_error", DEFAULT_STATIONARY_TOLERANCE, "Изменение комплексного поля"),
    ("contrast", DEFAULT_HOMOGENEOUS_CONTRAST, "Контраст интенсивности"),
)
for ax, (metric, threshold, title) in zip(axes, metrics):
    widths, amplitudes, values = rectangular_grid(
        [row["sigma"] for row in measurements],
        [row["A"] for row in measurements],
        [row[metric] for row in measurements],
    )
    ratio = values / threshold
    image = ax.pcolormesh(
        cell_edges(widths, logarithmic=True), cell_edges(amplitudes),
        np.maximum(ratio, np.finfo(float).tiny),
        norm=LogNorm(vmin=1e-3, vmax=1e3), cmap="RdBu_r", rasterized=True,
    )
    x, y = np.meshgrid(widths, amplitudes)
    ax.scatter(x.ravel(), y.ravel(), s=9, color="black", alpha=.5)
    ax.set(xscale="log", xlabel=r"Ширина $\sigma$", title=title)
    format_log_x(ax)
axes[0].set_ylabel(r"Амплитуда $A$")
colorbar = figure.colorbar(image, ax=axes, extend="both", ticks=[1e-3, 1e-2, 1e-1, 1, 10, 100, 1000])
colorbar.set_label("Значение / порог")
colorbar.ax.axhline(1, color="black", linewidth=1)
figure.suptitle(case_title + r"; $t \in [80,100]$")
save_figure(RESULTS_DIR, "measurement_maps_" + case, figure)
plt.show()

In [ ]:
widths, amplitudes, matrix = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [solutions.index(row["solution"]) for row in coarse],
)
figure, ax = plt.subplots(figsize=(6.4, 6.1))
image, nodes = plot_outcome_map(ax, widths, amplitudes, matrix, colors)
nodes.set(sizes=[10], color="black", alpha=.5)
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$", title=case_title + "; решения при $T=100$")
format_log_x(ax)
figure.legend(handles=[Patch(color=color, label=solution_labels[solution])
                       for color, solution in zip(colors, solutions)],
              loc="lower center", frameon=False)
figure.tight_layout(rect=(0, .20, 1, 1))
save_figure(RESULTS_DIR, "outcome_map_" + case, figure)
plt.show()

In [ ]:
energy_widths, energy_amplitudes, initial_energy = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [np.sqrt(np.pi) * row["A"]**2 * row["sigma"] for row in coarse],
)
energy_norm = LogNorm(vmin=initial_energy.min(), vmax=initial_energy.max())
energy_cmap = plt.get_cmap("viridis")
solution_markers = {
    SolutionKind.HOMOGENEOUS: "o",
    SolutionKind.SINGLE_SOLITON: "*",
    SolutionKind.OTHER: "^",
}
figure, ax = plt.subplots(figsize=(9, 8))
image = ax.pcolormesh(
    cell_edges(energy_widths, logarithmic=True), cell_edges(energy_amplitudes),
    initial_energy, cmap=energy_cmap, norm=energy_norm, rasterized=True,
)
for solution, marker in solution_markers.items():
    rows = [row for row in coarse if row["solution"] is solution]
    ax.scatter([row["sigma"] for row in rows], [row["A"] for row in rows],
               marker=marker, s=80 if marker == "*" else 35,
               facecolors="white", edgecolors="black", linewidths=.8,
               label=solution_labels[solution])
for row in coarse:
    energy = np.sqrt(np.pi) * row["A"]**2 * row["sigma"]
    red, green, blue, _ = energy_cmap(energy_norm(energy))
    luminance = .2126 * red + .7152 * green + .0722 * blue
    ax.annotate(f"{energy:.3g}", (row["sigma"], row["A"]),
                xytext=(0, -9), textcoords="offset points", ha="center", va="top",
                fontsize=9, color="white" if luminance < .5 else "black")
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$",
       title=case_title + r"; $E_0=\sqrt{\pi}A^2\sigma$")
format_log_x(ax)
colorbar = figure.colorbar(image, ax=ax, pad=.03)
colorbar.set_label(r"Начальная энергия $E_0$")
handles, labels = ax.get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", frameon=False, ncol=3, fontsize=10)
figure.tight_layout(rect=(0, .065, 1, 1))
save_figure(RESULTS_DIR, "initial_energy_map_" + case, figure)
plt.show()


In [ ]:
profile_times = (0., 20., 50., 100.)
profile_colors = ("#8b94a9", "#e69f00", "#0072b2", "#b2182b")
profile_styles = ("--", "-", "-", "-")
profile_amplitudes = sorted({row["A"] for row in coarse}, reverse=True)
profile_widths = sorted({row["sigma"] for row in coarse})
profiles_by_node = profiles_by_case[case]
assert len(profiles_by_node) == len(profile_amplitudes) * len(profile_widths)
maximum_intensity = max(profile.values.max() for profiles in profiles_by_node.values() for profile in profiles)

figure, axes = plt.subplots(
    len(profile_amplitudes), len(profile_widths), figsize=(20, 17),
    sharex=True, sharey=True, squeeze=False,
)
for row, amplitude in enumerate(profile_amplitudes):
    for column, width in enumerate(profile_widths):
        ax = axes[row, column]
        profiles = profiles_by_node[amplitude, width]
        for time, profile, color, style in zip(profile_times, profiles, profile_colors, profile_styles):
            ax.plot(profile.theta, profile.values, color=color, linestyle=style,
                    linewidth=1.5 if time == profile_times[-1] else .9, label=f"t={time:g}")
        ax.set_xlim(-np.pi, np.pi)
        ax.set_ylim(0, 1.05 * maximum_intensity)
        ax.set_xticks([-np.pi, 0, np.pi], labels=[r"$-\pi$", "0", r"$\pi$"])
        ax.tick_params(labelsize=8)
        ax.grid(alpha=.15)
        if row == 0:
            ax.set_title(rf"$\sigma={width:.3g}$", fontsize=11)
        if column == 0:
            ax.set_ylabel(rf"$A={amplitude:g}$", fontsize=11)
figure.supxlabel(r"Координата $\theta$", fontsize=13)
figure.supylabel(r"Интенсивность $|\psi|^2$; строки: амплитуда начального импульса", fontsize=13)
figure.suptitle(case_title + "; профили интенсивности", fontsize=17, y=.995)
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(.5, .975), ncol=len(profile_times), frameon=False)
figure.tight_layout(rect=(.02, .02, 1, .945), h_pad=.5, w_pad=.5)
save_figure(RESULTS_DIR, "intensity_profiles_grid_" + case, figure)
plt.show()

## F=1.8, Δ=3

Порядок рисунков: изменение поля и контраст; три категории решений; начальная энергия; профили при $t=0,20,50,100$. Цвета, маркеры и шкалы совпадают с экспериментом 1. Энергия вычисляется аналитически: $E_0=\sqrt{\pi}A^2\sigma$.


In [ ]:
case = 'F_lower'
case_title = 'F=1.8, Δ=3'
coarse = [row for row in map_results if row["case"] == case]
measurements = coarse

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True, layout="constrained")
metrics = (
    ("relative_field_l2_error", DEFAULT_STATIONARY_TOLERANCE, "Изменение комплексного поля"),
    ("contrast", DEFAULT_HOMOGENEOUS_CONTRAST, "Контраст интенсивности"),
)
for ax, (metric, threshold, title) in zip(axes, metrics):
    widths, amplitudes, values = rectangular_grid(
        [row["sigma"] for row in measurements],
        [row["A"] for row in measurements],
        [row[metric] for row in measurements],
    )
    ratio = values / threshold
    image = ax.pcolormesh(
        cell_edges(widths, logarithmic=True), cell_edges(amplitudes),
        np.maximum(ratio, np.finfo(float).tiny),
        norm=LogNorm(vmin=1e-3, vmax=1e3), cmap="RdBu_r", rasterized=True,
    )
    x, y = np.meshgrid(widths, amplitudes)
    ax.scatter(x.ravel(), y.ravel(), s=9, color="black", alpha=.5)
    ax.set(xscale="log", xlabel=r"Ширина $\sigma$", title=title)
    format_log_x(ax)
axes[0].set_ylabel(r"Амплитуда $A$")
colorbar = figure.colorbar(image, ax=axes, extend="both", ticks=[1e-3, 1e-2, 1e-1, 1, 10, 100, 1000])
colorbar.set_label("Значение / порог")
colorbar.ax.axhline(1, color="black", linewidth=1)
figure.suptitle(case_title + r"; $t \in [80,100]$")
save_figure(RESULTS_DIR, "measurement_maps_" + case, figure)
plt.show()

In [ ]:
widths, amplitudes, matrix = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [solutions.index(row["solution"]) for row in coarse],
)
figure, ax = plt.subplots(figsize=(6.4, 6.1))
image, nodes = plot_outcome_map(ax, widths, amplitudes, matrix, colors)
nodes.set(sizes=[10], color="black", alpha=.5)
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$", title=case_title + "; решения при $T=100$")
format_log_x(ax)
figure.legend(handles=[Patch(color=color, label=solution_labels[solution])
                       for color, solution in zip(colors, solutions)],
              loc="lower center", frameon=False)
figure.tight_layout(rect=(0, .20, 1, 1))
save_figure(RESULTS_DIR, "outcome_map_" + case, figure)
plt.show()

In [ ]:
energy_widths, energy_amplitudes, initial_energy = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [np.sqrt(np.pi) * row["A"]**2 * row["sigma"] for row in coarse],
)
energy_norm = LogNorm(vmin=initial_energy.min(), vmax=initial_energy.max())
energy_cmap = plt.get_cmap("viridis")
solution_markers = {
    SolutionKind.HOMOGENEOUS: "o",
    SolutionKind.SINGLE_SOLITON: "*",
    SolutionKind.OTHER: "^",
}
figure, ax = plt.subplots(figsize=(9, 8))
image = ax.pcolormesh(
    cell_edges(energy_widths, logarithmic=True), cell_edges(energy_amplitudes),
    initial_energy, cmap=energy_cmap, norm=energy_norm, rasterized=True,
)
for solution, marker in solution_markers.items():
    rows = [row for row in coarse if row["solution"] is solution]
    ax.scatter([row["sigma"] for row in rows], [row["A"] for row in rows],
               marker=marker, s=80 if marker == "*" else 35,
               facecolors="white", edgecolors="black", linewidths=.8,
               label=solution_labels[solution])
for row in coarse:
    energy = np.sqrt(np.pi) * row["A"]**2 * row["sigma"]
    red, green, blue, _ = energy_cmap(energy_norm(energy))
    luminance = .2126 * red + .7152 * green + .0722 * blue
    ax.annotate(f"{energy:.3g}", (row["sigma"], row["A"]),
                xytext=(0, -9), textcoords="offset points", ha="center", va="top",
                fontsize=9, color="white" if luminance < .5 else "black")
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$",
       title=case_title + r"; $E_0=\sqrt{\pi}A^2\sigma$")
format_log_x(ax)
colorbar = figure.colorbar(image, ax=ax, pad=.03)
colorbar.set_label(r"Начальная энергия $E_0$")
handles, labels = ax.get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", frameon=False, ncol=3, fontsize=10)
figure.tight_layout(rect=(0, .065, 1, 1))
save_figure(RESULTS_DIR, "initial_energy_map_" + case, figure)
plt.show()


In [ ]:
profile_times = (0., 20., 50., 100.)
profile_colors = ("#8b94a9", "#e69f00", "#0072b2", "#b2182b")
profile_styles = ("--", "-", "-", "-")
profile_amplitudes = sorted({row["A"] for row in coarse}, reverse=True)
profile_widths = sorted({row["sigma"] for row in coarse})
profiles_by_node = profiles_by_case[case]
assert len(profiles_by_node) == len(profile_amplitudes) * len(profile_widths)
maximum_intensity = max(profile.values.max() for profiles in profiles_by_node.values() for profile in profiles)

figure, axes = plt.subplots(
    len(profile_amplitudes), len(profile_widths), figsize=(20, 17),
    sharex=True, sharey=True, squeeze=False,
)
for row, amplitude in enumerate(profile_amplitudes):
    for column, width in enumerate(profile_widths):
        ax = axes[row, column]
        profiles = profiles_by_node[amplitude, width]
        for time, profile, color, style in zip(profile_times, profiles, profile_colors, profile_styles):
            ax.plot(profile.theta, profile.values, color=color, linestyle=style,
                    linewidth=1.5 if time == profile_times[-1] else .9, label=f"t={time:g}")
        ax.set_xlim(-np.pi, np.pi)
        ax.set_ylim(0, 1.05 * maximum_intensity)
        ax.set_xticks([-np.pi, 0, np.pi], labels=[r"$-\pi$", "0", r"$\pi$"])
        ax.tick_params(labelsize=8)
        ax.grid(alpha=.15)
        if row == 0:
            ax.set_title(rf"$\sigma={width:.3g}$", fontsize=11)
        if column == 0:
            ax.set_ylabel(rf"$A={amplitude:g}$", fontsize=11)
figure.supxlabel(r"Координата $\theta$", fontsize=13)
figure.supylabel(r"Интенсивность $|\psi|^2$; строки: амплитуда начального импульса", fontsize=13)
figure.suptitle(case_title + "; профили интенсивности", fontsize=17, y=.995)
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(.5, .975), ncol=len(profile_times), frameon=False)
figure.tight_layout(rect=(.02, .02, 1, .945), h_pad=.5, w_pad=.5)
save_figure(RESULTS_DIR, "intensity_profiles_grid_" + case, figure)
plt.show()

## F=2.2, Δ=3

Порядок рисунков: изменение поля и контраст; три категории решений; начальная энергия; профили при $t=0,20,50,100$. Цвета, маркеры и шкалы совпадают с экспериментом 1. Энергия вычисляется аналитически: $E_0=\sqrt{\pi}A^2\sigma$.


In [ ]:
case = 'F_higher'
case_title = 'F=2.2, Δ=3'
coarse = [row for row in map_results if row["case"] == case]
measurements = coarse

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True, layout="constrained")
metrics = (
    ("relative_field_l2_error", DEFAULT_STATIONARY_TOLERANCE, "Изменение комплексного поля"),
    ("contrast", DEFAULT_HOMOGENEOUS_CONTRAST, "Контраст интенсивности"),
)
for ax, (metric, threshold, title) in zip(axes, metrics):
    widths, amplitudes, values = rectangular_grid(
        [row["sigma"] for row in measurements],
        [row["A"] for row in measurements],
        [row[metric] for row in measurements],
    )
    ratio = values / threshold
    image = ax.pcolormesh(
        cell_edges(widths, logarithmic=True), cell_edges(amplitudes),
        np.maximum(ratio, np.finfo(float).tiny),
        norm=LogNorm(vmin=1e-3, vmax=1e3), cmap="RdBu_r", rasterized=True,
    )
    x, y = np.meshgrid(widths, amplitudes)
    ax.scatter(x.ravel(), y.ravel(), s=9, color="black", alpha=.5)
    ax.set(xscale="log", xlabel=r"Ширина $\sigma$", title=title)
    format_log_x(ax)
axes[0].set_ylabel(r"Амплитуда $A$")
colorbar = figure.colorbar(image, ax=axes, extend="both", ticks=[1e-3, 1e-2, 1e-1, 1, 10, 100, 1000])
colorbar.set_label("Значение / порог")
colorbar.ax.axhline(1, color="black", linewidth=1)
figure.suptitle(case_title + r"; $t \in [80,100]$")
save_figure(RESULTS_DIR, "measurement_maps_" + case, figure)
plt.show()

In [ ]:
widths, amplitudes, matrix = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [solutions.index(row["solution"]) for row in coarse],
)
figure, ax = plt.subplots(figsize=(6.4, 6.1))
image, nodes = plot_outcome_map(ax, widths, amplitudes, matrix, colors)
nodes.set(sizes=[10], color="black", alpha=.5)
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$", title=case_title + "; решения при $T=100$")
format_log_x(ax)
figure.legend(handles=[Patch(color=color, label=solution_labels[solution])
                       for color, solution in zip(colors, solutions)],
              loc="lower center", frameon=False)
figure.tight_layout(rect=(0, .20, 1, 1))
save_figure(RESULTS_DIR, "outcome_map_" + case, figure)
plt.show()

In [ ]:
energy_widths, energy_amplitudes, initial_energy = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [np.sqrt(np.pi) * row["A"]**2 * row["sigma"] for row in coarse],
)
energy_norm = LogNorm(vmin=initial_energy.min(), vmax=initial_energy.max())
energy_cmap = plt.get_cmap("viridis")
solution_markers = {
    SolutionKind.HOMOGENEOUS: "o",
    SolutionKind.SINGLE_SOLITON: "*",
    SolutionKind.OTHER: "^",
}
figure, ax = plt.subplots(figsize=(9, 8))
image = ax.pcolormesh(
    cell_edges(energy_widths, logarithmic=True), cell_edges(energy_amplitudes),
    initial_energy, cmap=energy_cmap, norm=energy_norm, rasterized=True,
)
for solution, marker in solution_markers.items():
    rows = [row for row in coarse if row["solution"] is solution]
    ax.scatter([row["sigma"] for row in rows], [row["A"] for row in rows],
               marker=marker, s=80 if marker == "*" else 35,
               facecolors="white", edgecolors="black", linewidths=.8,
               label=solution_labels[solution])
for row in coarse:
    energy = np.sqrt(np.pi) * row["A"]**2 * row["sigma"]
    red, green, blue, _ = energy_cmap(energy_norm(energy))
    luminance = .2126 * red + .7152 * green + .0722 * blue
    ax.annotate(f"{energy:.3g}", (row["sigma"], row["A"]),
                xytext=(0, -9), textcoords="offset points", ha="center", va="top",
                fontsize=9, color="white" if luminance < .5 else "black")
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$",
       title=case_title + r"; $E_0=\sqrt{\pi}A^2\sigma$")
format_log_x(ax)
colorbar = figure.colorbar(image, ax=ax, pad=.03)
colorbar.set_label(r"Начальная энергия $E_0$")
handles, labels = ax.get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", frameon=False, ncol=3, fontsize=10)
figure.tight_layout(rect=(0, .065, 1, 1))
save_figure(RESULTS_DIR, "initial_energy_map_" + case, figure)
plt.show()


In [ ]:
profile_times = (0., 20., 50., 100.)
profile_colors = ("#8b94a9", "#e69f00", "#0072b2", "#b2182b")
profile_styles = ("--", "-", "-", "-")
profile_amplitudes = sorted({row["A"] for row in coarse}, reverse=True)
profile_widths = sorted({row["sigma"] for row in coarse})
profiles_by_node = profiles_by_case[case]
assert len(profiles_by_node) == len(profile_amplitudes) * len(profile_widths)
maximum_intensity = max(profile.values.max() for profiles in profiles_by_node.values() for profile in profiles)

figure, axes = plt.subplots(
    len(profile_amplitudes), len(profile_widths), figsize=(20, 17),
    sharex=True, sharey=True, squeeze=False,
)
for row, amplitude in enumerate(profile_amplitudes):
    for column, width in enumerate(profile_widths):
        ax = axes[row, column]
        profiles = profiles_by_node[amplitude, width]
        for time, profile, color, style in zip(profile_times, profiles, profile_colors, profile_styles):
            ax.plot(profile.theta, profile.values, color=color, linestyle=style,
                    linewidth=1.5 if time == profile_times[-1] else .9, label=f"t={time:g}")
        ax.set_xlim(-np.pi, np.pi)
        ax.set_ylim(0, 1.05 * maximum_intensity)
        ax.set_xticks([-np.pi, 0, np.pi], labels=[r"$-\pi$", "0", r"$\pi$"])
        ax.tick_params(labelsize=8)
        ax.grid(alpha=.15)
        if row == 0:
            ax.set_title(rf"$\sigma={width:.3g}$", fontsize=11)
        if column == 0:
            ax.set_ylabel(rf"$A={amplitude:g}$", fontsize=11)
figure.supxlabel(r"Координата $\theta$", fontsize=13)
figure.supylabel(r"Интенсивность $|\psi|^2$; строки: амплитуда начального импульса", fontsize=13)
figure.suptitle(case_title + "; профили интенсивности", fontsize=17, y=.995)
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(.5, .975), ncol=len(profile_times), frameon=False)
figure.tight_layout(rect=(.02, .02, 1, .945), h_pad=.5, w_pad=.5)
save_figure(RESULTS_DIR, "intensity_profiles_grid_" + case, figure)
plt.show()

## F=2, Δ=2.5

Порядок рисунков: изменение поля и контраст; три категории решений; начальная энергия; профили при $t=0,20,50,100$. Цвета, маркеры и шкалы совпадают с экспериментом 1. Энергия вычисляется аналитически: $E_0=\sqrt{\pi}A^2\sigma$.


In [ ]:
case = 'delta_lower'
case_title = 'F=2, Δ=2.5'
coarse = [row for row in map_results if row["case"] == case]
measurements = coarse

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True, layout="constrained")
metrics = (
    ("relative_field_l2_error", DEFAULT_STATIONARY_TOLERANCE, "Изменение комплексного поля"),
    ("contrast", DEFAULT_HOMOGENEOUS_CONTRAST, "Контраст интенсивности"),
)
for ax, (metric, threshold, title) in zip(axes, metrics):
    widths, amplitudes, values = rectangular_grid(
        [row["sigma"] for row in measurements],
        [row["A"] for row in measurements],
        [row[metric] for row in measurements],
    )
    ratio = values / threshold
    image = ax.pcolormesh(
        cell_edges(widths, logarithmic=True), cell_edges(amplitudes),
        np.maximum(ratio, np.finfo(float).tiny),
        norm=LogNorm(vmin=1e-3, vmax=1e3), cmap="RdBu_r", rasterized=True,
    )
    x, y = np.meshgrid(widths, amplitudes)
    ax.scatter(x.ravel(), y.ravel(), s=9, color="black", alpha=.5)
    ax.set(xscale="log", xlabel=r"Ширина $\sigma$", title=title)
    format_log_x(ax)
axes[0].set_ylabel(r"Амплитуда $A$")
colorbar = figure.colorbar(image, ax=axes, extend="both", ticks=[1e-3, 1e-2, 1e-1, 1, 10, 100, 1000])
colorbar.set_label("Значение / порог")
colorbar.ax.axhline(1, color="black", linewidth=1)
figure.suptitle(case_title + r"; $t \in [80,100]$")
save_figure(RESULTS_DIR, "measurement_maps_" + case, figure)
plt.show()

In [ ]:
widths, amplitudes, matrix = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [solutions.index(row["solution"]) for row in coarse],
)
figure, ax = plt.subplots(figsize=(6.4, 6.1))
image, nodes = plot_outcome_map(ax, widths, amplitudes, matrix, colors)
nodes.set(sizes=[10], color="black", alpha=.5)
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$", title=case_title + "; решения при $T=100$")
format_log_x(ax)
figure.legend(handles=[Patch(color=color, label=solution_labels[solution])
                       for color, solution in zip(colors, solutions)],
              loc="lower center", frameon=False)
figure.tight_layout(rect=(0, .20, 1, 1))
save_figure(RESULTS_DIR, "outcome_map_" + case, figure)
plt.show()

In [ ]:
energy_widths, energy_amplitudes, initial_energy = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [np.sqrt(np.pi) * row["A"]**2 * row["sigma"] for row in coarse],
)
energy_norm = LogNorm(vmin=initial_energy.min(), vmax=initial_energy.max())
energy_cmap = plt.get_cmap("viridis")
solution_markers = {
    SolutionKind.HOMOGENEOUS: "o",
    SolutionKind.SINGLE_SOLITON: "*",
    SolutionKind.OTHER: "^",
}
figure, ax = plt.subplots(figsize=(9, 8))
image = ax.pcolormesh(
    cell_edges(energy_widths, logarithmic=True), cell_edges(energy_amplitudes),
    initial_energy, cmap=energy_cmap, norm=energy_norm, rasterized=True,
)
for solution, marker in solution_markers.items():
    rows = [row for row in coarse if row["solution"] is solution]
    ax.scatter([row["sigma"] for row in rows], [row["A"] for row in rows],
               marker=marker, s=80 if marker == "*" else 35,
               facecolors="white", edgecolors="black", linewidths=.8,
               label=solution_labels[solution])
for row in coarse:
    energy = np.sqrt(np.pi) * row["A"]**2 * row["sigma"]
    red, green, blue, _ = energy_cmap(energy_norm(energy))
    luminance = .2126 * red + .7152 * green + .0722 * blue
    ax.annotate(f"{energy:.3g}", (row["sigma"], row["A"]),
                xytext=(0, -9), textcoords="offset points", ha="center", va="top",
                fontsize=9, color="white" if luminance < .5 else "black")
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$",
       title=case_title + r"; $E_0=\sqrt{\pi}A^2\sigma$")
format_log_x(ax)
colorbar = figure.colorbar(image, ax=ax, pad=.03)
colorbar.set_label(r"Начальная энергия $E_0$")
handles, labels = ax.get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", frameon=False, ncol=3, fontsize=10)
figure.tight_layout(rect=(0, .065, 1, 1))
save_figure(RESULTS_DIR, "initial_energy_map_" + case, figure)
plt.show()


In [ ]:
profile_times = (0., 20., 50., 100.)
profile_colors = ("#8b94a9", "#e69f00", "#0072b2", "#b2182b")
profile_styles = ("--", "-", "-", "-")
profile_amplitudes = sorted({row["A"] for row in coarse}, reverse=True)
profile_widths = sorted({row["sigma"] for row in coarse})
profiles_by_node = profiles_by_case[case]
assert len(profiles_by_node) == len(profile_amplitudes) * len(profile_widths)
maximum_intensity = max(profile.values.max() for profiles in profiles_by_node.values() for profile in profiles)

figure, axes = plt.subplots(
    len(profile_amplitudes), len(profile_widths), figsize=(20, 17),
    sharex=True, sharey=True, squeeze=False,
)
for row, amplitude in enumerate(profile_amplitudes):
    for column, width in enumerate(profile_widths):
        ax = axes[row, column]
        profiles = profiles_by_node[amplitude, width]
        for time, profile, color, style in zip(profile_times, profiles, profile_colors, profile_styles):
            ax.plot(profile.theta, profile.values, color=color, linestyle=style,
                    linewidth=1.5 if time == profile_times[-1] else .9, label=f"t={time:g}")
        ax.set_xlim(-np.pi, np.pi)
        ax.set_ylim(0, 1.05 * maximum_intensity)
        ax.set_xticks([-np.pi, 0, np.pi], labels=[r"$-\pi$", "0", r"$\pi$"])
        ax.tick_params(labelsize=8)
        ax.grid(alpha=.15)
        if row == 0:
            ax.set_title(rf"$\sigma={width:.3g}$", fontsize=11)
        if column == 0:
            ax.set_ylabel(rf"$A={amplitude:g}$", fontsize=11)
figure.supxlabel(r"Координата $\theta$", fontsize=13)
figure.supylabel(r"Интенсивность $|\psi|^2$; строки: амплитуда начального импульса", fontsize=13)
figure.suptitle(case_title + "; профили интенсивности", fontsize=17, y=.995)
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(.5, .975), ncol=len(profile_times), frameon=False)
figure.tight_layout(rect=(.02, .02, 1, .945), h_pad=.5, w_pad=.5)
save_figure(RESULTS_DIR, "intensity_profiles_grid_" + case, figure)
plt.show()

## F=2, Δ=3.5

Порядок рисунков: изменение поля и контраст; три категории решений; начальная энергия; профили при $t=0,20,50,100$. Цвета, маркеры и шкалы совпадают с экспериментом 1. Энергия вычисляется аналитически: $E_0=\sqrt{\pi}A^2\sigma$.


In [ ]:
case = 'delta_higher'
case_title = 'F=2, Δ=3.5'
coarse = [row for row in map_results if row["case"] == case]
measurements = coarse

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True, layout="constrained")
metrics = (
    ("relative_field_l2_error", DEFAULT_STATIONARY_TOLERANCE, "Изменение комплексного поля"),
    ("contrast", DEFAULT_HOMOGENEOUS_CONTRAST, "Контраст интенсивности"),
)
for ax, (metric, threshold, title) in zip(axes, metrics):
    widths, amplitudes, values = rectangular_grid(
        [row["sigma"] for row in measurements],
        [row["A"] for row in measurements],
        [row[metric] for row in measurements],
    )
    ratio = values / threshold
    image = ax.pcolormesh(
        cell_edges(widths, logarithmic=True), cell_edges(amplitudes),
        np.maximum(ratio, np.finfo(float).tiny),
        norm=LogNorm(vmin=1e-3, vmax=1e3), cmap="RdBu_r", rasterized=True,
    )
    x, y = np.meshgrid(widths, amplitudes)
    ax.scatter(x.ravel(), y.ravel(), s=9, color="black", alpha=.5)
    ax.set(xscale="log", xlabel=r"Ширина $\sigma$", title=title)
    format_log_x(ax)
axes[0].set_ylabel(r"Амплитуда $A$")
colorbar = figure.colorbar(image, ax=axes, extend="both", ticks=[1e-3, 1e-2, 1e-1, 1, 10, 100, 1000])
colorbar.set_label("Значение / порог")
colorbar.ax.axhline(1, color="black", linewidth=1)
figure.suptitle(case_title + r"; $t \in [80,100]$")
save_figure(RESULTS_DIR, "measurement_maps_" + case, figure)
plt.show()

In [ ]:
widths, amplitudes, matrix = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [solutions.index(row["solution"]) for row in coarse],
)
figure, ax = plt.subplots(figsize=(6.4, 6.1))
image, nodes = plot_outcome_map(ax, widths, amplitudes, matrix, colors)
nodes.set(sizes=[10], color="black", alpha=.5)
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$", title=case_title + "; решения при $T=100$")
format_log_x(ax)
figure.legend(handles=[Patch(color=color, label=solution_labels[solution])
                       for color, solution in zip(colors, solutions)],
              loc="lower center", frameon=False)
figure.tight_layout(rect=(0, .20, 1, 1))
save_figure(RESULTS_DIR, "outcome_map_" + case, figure)
plt.show()

In [ ]:
energy_widths, energy_amplitudes, initial_energy = rectangular_grid(
    [row["sigma"] for row in coarse], [row["A"] for row in coarse],
    [np.sqrt(np.pi) * row["A"]**2 * row["sigma"] for row in coarse],
)
energy_norm = LogNorm(vmin=initial_energy.min(), vmax=initial_energy.max())
energy_cmap = plt.get_cmap("viridis")
solution_markers = {
    SolutionKind.HOMOGENEOUS: "o",
    SolutionKind.SINGLE_SOLITON: "*",
    SolutionKind.OTHER: "^",
}
figure, ax = plt.subplots(figsize=(9, 8))
image = ax.pcolormesh(
    cell_edges(energy_widths, logarithmic=True), cell_edges(energy_amplitudes),
    initial_energy, cmap=energy_cmap, norm=energy_norm, rasterized=True,
)
for solution, marker in solution_markers.items():
    rows = [row for row in coarse if row["solution"] is solution]
    ax.scatter([row["sigma"] for row in rows], [row["A"] for row in rows],
               marker=marker, s=80 if marker == "*" else 35,
               facecolors="white", edgecolors="black", linewidths=.8,
               label=solution_labels[solution])
for row in coarse:
    energy = np.sqrt(np.pi) * row["A"]**2 * row["sigma"]
    red, green, blue, _ = energy_cmap(energy_norm(energy))
    luminance = .2126 * red + .7152 * green + .0722 * blue
    ax.annotate(f"{energy:.3g}", (row["sigma"], row["A"]),
                xytext=(0, -9), textcoords="offset points", ha="center", va="top",
                fontsize=9, color="white" if luminance < .5 else "black")
ax.set(xscale="log", xlabel=r"Ширина $\sigma$", ylabel=r"Амплитуда $A$",
       title=case_title + r"; $E_0=\sqrt{\pi}A^2\sigma$")
format_log_x(ax)
colorbar = figure.colorbar(image, ax=ax, pad=.03)
colorbar.set_label(r"Начальная энергия $E_0$")
handles, labels = ax.get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", frameon=False, ncol=3, fontsize=10)
figure.tight_layout(rect=(0, .065, 1, 1))
save_figure(RESULTS_DIR, "initial_energy_map_" + case, figure)
plt.show()


In [ ]:
profile_times = (0., 20., 50., 100.)
profile_colors = ("#8b94a9", "#e69f00", "#0072b2", "#b2182b")
profile_styles = ("--", "-", "-", "-")
profile_amplitudes = sorted({row["A"] for row in coarse}, reverse=True)
profile_widths = sorted({row["sigma"] for row in coarse})
profiles_by_node = profiles_by_case[case]
assert len(profiles_by_node) == len(profile_amplitudes) * len(profile_widths)
maximum_intensity = max(profile.values.max() for profiles in profiles_by_node.values() for profile in profiles)

figure, axes = plt.subplots(
    len(profile_amplitudes), len(profile_widths), figsize=(20, 17),
    sharex=True, sharey=True, squeeze=False,
)
for row, amplitude in enumerate(profile_amplitudes):
    for column, width in enumerate(profile_widths):
        ax = axes[row, column]
        profiles = profiles_by_node[amplitude, width]
        for time, profile, color, style in zip(profile_times, profiles, profile_colors, profile_styles):
            ax.plot(profile.theta, profile.values, color=color, linestyle=style,
                    linewidth=1.5 if time == profile_times[-1] else .9, label=f"t={time:g}")
        ax.set_xlim(-np.pi, np.pi)
        ax.set_ylim(0, 1.05 * maximum_intensity)
        ax.set_xticks([-np.pi, 0, np.pi], labels=[r"$-\pi$", "0", r"$\pi$"])
        ax.tick_params(labelsize=8)
        ax.grid(alpha=.15)
        if row == 0:
            ax.set_title(rf"$\sigma={width:.3g}$", fontsize=11)
        if column == 0:
            ax.set_ylabel(rf"$A={amplitude:g}$", fontsize=11)
figure.supxlabel(r"Координата $\theta$", fontsize=13)
figure.supylabel(r"Интенсивность $|\psi|^2$; строки: амплитуда начального импульса", fontsize=13)
figure.suptitle(case_title + "; профили интенсивности", fontsize=17, y=.995)
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(.5, .975), ncol=len(profile_times), frameon=False)
figure.tight_layout(rect=(.02, .02, 1, .945), h_pad=.5, w_pad=.5)
save_figure(RESULTS_DIR, "intensity_profiles_grid_" + case, figure)
plt.show()

## Сравнение количества решений

Считаем узлы каждой категории; эти количества не являются площадями бассейнов притяжения.


In [ ]:
counts_by_case = [Counter(row["solution"] for row in map_results if row["case"] == case) for case in case_order]
values = np.array([[counts[solution] for counts in counts_by_case] for solution in solutions])
coordinates = np.arange(len(case_order))
tick_labels = []
for case in case_order:
    row = next(row for row in map_results if row["case"] == case)
    tick_labels.append(f'F={row["F"]:g}\nΔ={row["delta"]:g}')
figure, ax = plt.subplots(figsize=(8, 5.5))
bars = plot_stacked_counts(ax, coordinates, values)
for group, solution, color in zip(bars, solutions, colors):
    group.set_label(solution_labels[solution])
    for bar in group:
        bar.set(color=color, linewidth=0)
        if bar.get_height():
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+bar.get_height()/2,
                    str(int(bar.get_height())), ha="center", va="center",
                    color="white" if solution is SolutionKind.HOMOGENEOUS else "black")
ax.set_xticks(coordinates, labels=tick_labels)
ax.set(ylabel="Число узлов из 64", ylim=(0, 64), title="Решения при T=100")
figure.legend(loc="lower center", frameon=False)
figure.tight_layout(rect=(0, .18, 1, 1))
save_figure(RESULTS_DIR, "solution_counts_T100", figure)
plt.show()